# Frozen estimator schema and mutation tests

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import unittest,tempfile,json
from pathlib import Path
import numpy as np
from environment_adapter import BaselineState
from baseline_rewards import TaskReward,MaxSupportReward
from preference_scoring import load_preference_model,PreferenceModel
class Checks(unittest.TestCase):
    def test_schema_and_clipping(self):
        model=load_preference_model();x=np.full((2,54),1e6)
        expected=np.mean([m.predict_proba(np.clip(s.transform(x),0,1))[:,0] for m,s in zip(model.models,model.scalers)],axis=0)
        np.testing.assert_allclose(model.predict_q(x),expected)
        with self.assertRaises(ValueError):model.predict_q(np.zeros((1,2)))
        with self.assertRaises(ValueError):model.predict_q(np.full((1,54),np.nan))
        with self.assertRaises(ValueError):PreferenceModel([],[])
    def test_model_and_scaler_mutation_rejected(self):
        for scaler in [False,True]:
            model=load_preference_model()
            if scaler:model.scalers[0].scale_[0]+=1
            else:model.models[0].coef_[0,0]+=1
            with self.assertRaises(RuntimeError):model.predict_q(np.zeros((1,54)))
print('Frozen estimator schema and mutation tests definitions/execution completed.')


Frozen increasing-preference scoring definitions/execution completed.
Task progress and fresh-pair rewards definitions/execution completed.
Matched observation action and window adapter definitions/execution completed.
Frozen estimator schema and mutation tests definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Tests run:',result.testsRun)

test_model_and_scaler_mutation_rejected (__main__.Checks) ... 

Frozen runtime contract definitions/execution completed.


ok


test_schema_and_clipping (__main__.Checks) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.544s

OK


Tests run: 2
